#  2 boyut Değerlendirme

1. Retrieval

- Doğru Chunk bulundu mu?

2. Generation

- Cevap doğru mu?

# Retrieval Metrics

- Metrik  -   Ne ölçer?

- Precission@K ----  İlk K'dan kaçı ilgili

- Recall@K. ----   Tüm ilgilerden kaçı K'da

- MRR    -----       Sorgular boyunca, ilk ilgili sonucun sıra- tersini ortalaması

- nDCG  ----    Rank-aware score (ileri / farkındalık)

# Generation Metrics

- Faithfulness: cevap context'te destekleniyor mu? (hallunication yok mu)

- Answer Relevance: cevap soruyu cevaplıyor mu?

- Context Relevance: getirilen context soruyla ilgili mi?

- Context Utilization: context ne kadar kullanıldı?


# RAGAS (Farkındalık)

- RAGAS = metrikler otomatik hesaplayan bir LLM-as-judge değerlendirme aracı
Faithfulness, answer,reşevancy, context recall gibi metrikler tek çağrıda üretilir

# Ground Truth Dataset

- Manuel: domain expert yazar (en iyi)

- LLM-generated: GPT-4 ile synthetic Q&A üret (hizlı)

- qa = gen_prompt = """
Sepetix yardım metninden 3 Q&A çiftçi üret:
{context}

JSON liste olarak dön."""

-  synthetic_qa = qa_chain.invoke({"context": sepetix_doc})

- Örn: ("Sepetix Plus üyelik kaç TL?", "Üyelik & Sadakat")

# A/B Test Pattern

- for k in [1,2,3]:
-    hr, mrr = degerlendir(k = k)
-    print(f"k = {k}: hit-rate = {hr:.0%}. MRR = {mrr:.2f}")

# Production'da Değerlendirme

- Production'da değerlendirme süreklidir: monitoring araçları (LangSmith, Langfuse) çağrıları izler, kullanıcı geri bildirimi toplam - bu ML-engineer / MLOps tarafıdır, farkında ol

# Özet

- Retrieval: Precision/Recall/MRR @K

- Generation: Faithfulness, Relevance

- RAGAS -- otomatik değerlendirme aracı (farkındalık)

- A/B + ground truth = production pipeline

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

In [ ]:
docs = DirectoryLoader("sirket-bilgi-bankasi/", glob="**/*.md", loader_cls=TextLoader).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60).split_documents(docs)
KATEGORI = {
    "kargo-teslimat": "Kargo & Teslimat", "iade-degisim": "İade & Değişim",
    "siparis-odeme": "Sipariş & Ödeme", "uyelik-sadakat": "Üyelik & Sadakat",
    "sss": "Sıkça Sorulan Sorular",
}
for ch in chunks:
    ad = os.path.basename(ch.metadata["source"]).replace(".md", "")
    ch.metadata["kategori"] = KATEGORI.get(ad, ad)
load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectordb = Chroma.from_documents(chunks, embedding=embeddings)
print(len(chunks), "chunk")

In [ ]:
gold = [
    ("Ürünü kaç gün içinde iade edebilirim?", "İade & Değişim"),
    ("İade ettiğim paranın iadesi ne zaman?", "İade & Değişim"),
    ("Kargo ne zaman ücretsiz olur?", "Kargo & Teslimat"),
    ("Aynı gün teslimat var mı?", "Kargo & Teslimat"),
    ("Kapıda ödeme ek ücretli mi?", "Sipariş & Ödeme"),
    ("Sepetix Plus yıllık kaç TL?", "Üyelik & Sadakat"),
    ("Şifremi unuttum, ne yapmalıyım?", "Sıkça Sorulan Sorular"),
]
print(len(gold), "soru")

In [ ]:
def degerlendir(k=2):
    hit, rr = 0, 0.0
    for soru, dogru in gold:
        getirilen = [d.metadata["kategori"] for d in vectordb.similarity_search(soru, k=k)]
        if dogru in getirilen:
            hit += 1
            rr += 1 / (getirilen.index(dogru) + 1)
    n = len(gold)
    return hit / n, rr / n

hr, mrr = degerlendir(k=2)
print(f"Hit-rate@2: {hr:.0%} ({round(hr*len(gold))}/{len(gold)})  |  MRR: {mrr:.2f}")

In [ ]:
for soru, dogru in gold:
    getirilen = [d.metadata["kategori"] for d in vectordb.similarity_search(soru, k=2)]
    print(f"{'✓' if dogru in getirilen else 'x'} '{soru[:38]}' -> beklenen [{dogru}]")

In [ ]:
for k in [1, 2, 3]:
    hr, mrr = degerlendir(k=k)
    print(f"k={k}  hit-rate={hr:.0%} ({round(hr*len(gold))}/{len(gold)})  MRR={mrr:.2f}")

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()  # .env: OPENAI_API_KEY=sk-...
hakem = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

def faithfulness(baglam, cevap):
    p = f"""Aşağıdaki CEVAP yalnızca BAĞLAM ile destekleniyor mu? 1 (uydurma) - 5 (tam destekli), tek rakam
BAĞLAM: {baglam}
CEVAP: {cevap}
PUAN:"""
    return hakem.invoke(p).content.strip()

baglam = vectordb.similarity_search("Sepetix Plus kaç TL?", k=1)[0].page_content
print("Sadık cevap ->", faithfulness(baglam, "Sepetix Plus yıllık 199 TL'dir."))
print("Uydurma cevap ->", faithfulness(baglam, "Sepetix Plus aylık 5 TL'dir."))